# Quadtree Tile Index

Reads image pair labels, selects the coarsest TIFF pyramid page that still covers the CNN input dimensions at each quadtree depth, and writes one entry per `(pair, depth, x, y)` tile to `quadtree_annotations.json`.

In [37]:
from pathlib import Path
import json
import sys

import numpy as np
import tifffile
from PIL import Image

sys.path.append(str(Path.cwd().parent))

import conf
from setup.background_filter import (
    FilterConfig,
    PRODUCTION_CONFIG,
    TEST_RUN_CONFIGS,
    is_background_tile,
)
from setup.pair_mask import (
    PRODUCTION_MIN_INSIDE_FRACTION,
    TEST_RUN_AREA_THRESHOLDS,
    is_tile_excluded_by_mask,
    load_pair_mask,
    mask_for_page,
)


TEST_RUN = False
TEST_RUN_TILE_LIMIT = 100

IMAGE_DIR = conf.IMAGE_DIR
LABELS_PATH = conf.LABELS_PATH
ANNOTATION_PATH = conf.ANNOTATION_PATH

WSI_PAGES = conf.WSI_PAGES
CNN_INPUT_HEIGHT = conf.CNN_INPUT_HEIGHT
CNN_INPUT_WIDTH = conf.CNN_INPUT_WIDTH
MAX_CROP_DEPTH = conf.MAX_CROP_DEPTH

print(f"System   : {conf.SYSTEM_PREFIX}")
print(f"Test run : {TEST_RUN}")
print(f"Labels   : {LABELS_PATH}")
print(f"Output   : {ANNOTATION_PATH}")

System   : macos
Test run : True
Labels   : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_labels.json
Output   : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_quadtree_annotations.json


In [38]:
from setup.background_filter import is_background_tile

In [39]:
def matrix_dict_to_list(m):
    return [
        [m["t_00"], m["t_01"], m["t_02"]],
        [m["t_10"], m["t_11"], m["t_12"]],
        [m["t_20"], m["t_21"], m["t_22"]],
    ]


def load_pairs_from_labels(labels_path, image_dir):
    with open(labels_path, "r", encoding="utf-8") as f:
        labels = json.load(f)

    pairs = []

    for pair_id, item in enumerate(labels):
        source_id = item["source_image_id"]
        target_id = item["target_image_id"]

        source_path = image_dir / f"{source_id}.data"
        target_path = image_dir / f"{target_id}.data"

        if not source_path.exists():
            raise FileNotFoundError(source_path)

        if not target_path.exists():
            raise FileNotFoundError(target_path)

        pairs.append({
            "pair_id": pair_id,
            "moving_path": conf.image_relpath(source_id),
            "fixed_path": conf.image_relpath(target_id),
            "source_image_id": source_id,
            "target_image_id": target_id,
            "registration_error": item["registration_error"],
            "transformation_matrix": matrix_dict_to_list(item["transformation_matrix"]),
        })

    return pairs

In [40]:
def choose_pair_pyramid_page(fixed_path, moving_path, crop_depth, input_height=CNN_INPUT_HEIGHT, input_width=CNN_INPUT_WIDTH):
    fixed_path = conf.resolve(fixed_path)
    moving_path = conf.resolve(moving_path)
    grid = 2 ** crop_depth

    with tifffile.TiffFile(fixed_path) as fixed_slide, tifffile.TiffFile(moving_path) as moving_slide:
        candidates = []

        for page_idx in WSI_PAGES:
            fixed_h, fixed_w = fixed_slide.pages[page_idx].shape[:2]
            moving_h, moving_w = moving_slide.pages[page_idx].shape[:2]

            fixed_tile_h = fixed_h // grid
            fixed_tile_w = fixed_w // grid

            moving_tile_h = moving_h // grid
            moving_tile_w = moving_w // grid

            min_tile_h = min(fixed_tile_h, moving_tile_h)
            min_tile_w = min(fixed_tile_w, moving_tile_w)

            if min_tile_h >= input_height and min_tile_w >= input_width:
                candidates.append((page_idx, min_tile_h, min_tile_w))

    if not candidates:
        return None

    return candidates[-1]

In [41]:
def he_tile_excluded(pair, job_fields, page_cache, threshold_or_config):
    pyramid_page_idx = job_fields["pyramid_page_idx"]
    grid = job_fields["grid"]
    x_idx = job_fields["x_idx"]
    y_idx = job_fields["y_idx"]
    path = conf.resolve(pair["fixed_path"])
    key = (str(path), pyramid_page_idx)
    if key not in page_cache:
        with tifffile.TiffFile(path) as slide:
            page_cache[key] = slide.pages[pyramid_page_idx].asarray()
    page = page_cache[key]
    page_h, page_w = page.shape[:2]
    mask = mask_for_page(pair["pair_id"], page_w, page_h, pyramid_page_idx)
    if mask is not None:
        if isinstance(threshold_or_config, FilterConfig):
            threshold = PRODUCTION_MIN_INSIDE_FRACTION
        else:
            threshold = threshold_or_config
        return is_tile_excluded_by_mask(mask, grid, x_idx, y_idx, threshold)
    return is_background_tile(
        pair["fixed_path"], pyramid_page_idx,
        grid, x_idx, y_idx, page_cache,
        config=threshold_or_config,
    )


def build_quadtree_index(
    pairs,
    max_crop_depth,
    filter_config=PRODUCTION_CONFIG,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH,
    max_tiles=None,
):
    tile_jobs = []
    page_cache = {}

    for pair in pairs:
        for crop_depth in range(max_crop_depth + 1):
            chosen = choose_pair_pyramid_page(
                fixed_path=pair["fixed_path"],
                moving_path=pair["moving_path"],
                crop_depth=crop_depth,
                input_height=input_height,
                input_width=input_width,
            )

            if chosen is None:
                continue

            pyramid_page_idx, tile_h, tile_w = chosen
            grid = 2 ** crop_depth

            for y_idx in range(grid):
                for x_idx in range(grid):
                    job = {
                        "pair_id": pair["pair_id"],
                        "fixed_path": pair["fixed_path"],
                        "moving_path": pair["moving_path"],
                        "source_image_id": pair["source_image_id"],
                        "target_image_id": pair["target_image_id"],
                        "crop_depth": crop_depth,
                        "grid": grid,
                        "x_idx": x_idx,
                        "y_idx": y_idx,
                        "pyramid_page_idx": pyramid_page_idx,
                        "tile_h": tile_h,
                        "tile_w": tile_w,
                        "cnn_input_height": input_height,
                        "cnn_input_width": input_width,
                        "registration_error": pair["registration_error"],
                        "transformation_matrix": pair["transformation_matrix"],
                    }
                    if filter_config is not None:
                        job["binary_mask_excluded"] = he_tile_excluded(
                            pair, job, page_cache, filter_config,
                        )
                    tile_jobs.append(job)
                    if max_tiles is not None and len(tile_jobs) >= max_tiles:
                        return tile_jobs

    return tile_jobs


def save_json(data, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

In [42]:
pairs = load_pairs_from_labels(LABELS_PATH, IMAGE_DIR)

if TEST_RUN:
    first_pair_id = min(pair["pair_id"] for pair in pairs)
    pairs = [pair for pair in pairs if pair["pair_id"] == first_pair_id]
    print(f"TEST_RUN: pair_id {first_pair_id}, limit {TEST_RUN_TILE_LIMIT} tiles × {len(TEST_RUN_AREA_THRESHOLDS)} configs")
    if load_pair_mask(first_pair_id) is None:
        print("WARNING: no HE mask JSON — using texture heuristic. Run setup/draw_pair_mask.py first.")

    base_jobs = build_quadtree_index(
        pairs=pairs,
        max_crop_depth=MAX_CROP_DEPTH,
        filter_config=None,
        input_height=CNN_INPUT_HEIGHT,
        input_width=CNN_INPUT_WIDTH,
        max_tiles=TEST_RUN_TILE_LIMIT,
    )

    tile_jobs = []
    page_cache = {}
    class_excluded = {name: 0 for name in TEST_RUN_AREA_THRESHOLDS}

    for job in base_jobs:
        pair = pairs[0]
        for class_name, threshold in TEST_RUN_AREA_THRESHOLDS.items():
            use_mask = load_pair_mask(pair["pair_id"]) is not None
            cfg = threshold if use_mask else TEST_RUN_CONFIGS[class_name]
            fixed_bg = he_tile_excluded(pair, job, page_cache, cfg)
            if fixed_bg:
                class_excluded[class_name] += 1
            tile_jobs.append({
                **job,
                "binary_mask_excluded": fixed_bg,
                "test_run_class": class_name,
            })

    print("Excluded per config:")
    for class_name in sorted(class_excluded):
        n = class_excluded[class_name]
        print(f"  {class_name}: {n}/{len(base_jobs)} background")
else:
    tile_jobs = build_quadtree_index(
        pairs=pairs,
        max_crop_depth=MAX_CROP_DEPTH,
        filter_config=PRODUCTION_CONFIG,
        input_height=CNN_INPUT_HEIGHT,
        input_width=CNN_INPUT_WIDTH,
    )

save_json(tile_jobs, ANNOTATION_PATH)

print(f"pairs     : {len(pairs)}")
print(f"tile jobs : {len(tile_jobs)}")
print(f"saved to  : {ANNOTATION_PATH}")

if TEST_RUN:
    print("WARNING: TEST_RUN overwrites annotation JSON with multi-config test jobs only.")

print("\nFirst job:")
print(tile_jobs[0])

print("\nLast job:")
print(tile_jobs[-1])

TEST_RUN: pair_id 0, limit 100 tiles × 5 configs
Excluded per config:
  cfg_0: 13/100 background
  cfg_1: 7/100 background
  cfg_2: 6/100 background
  cfg_3: 0/100 background
  cfg_4: 0/100 background
pairs     : 1
tile jobs : 500
saved to  : /Users/alexanderhallmann/Desktop/medical-image-registration/data/macos_quadtree_annotations.json

First job:
{'pair_id': 0, 'fixed_path': 'data/images/6045.data', 'moving_path': 'data/images/6036.data', 'source_image_id': 6036, 'target_image_id': 6045, 'crop_depth': 0, 'grid': 1, 'x_idx': 0, 'y_idx': 0, 'pyramid_page_idx': 4, 'tile_h': 1388, 'tile_w': 2054, 'cnn_input_height': 344, 'cnn_input_width': 512, 'registration_error': 104.38675487149722, 'transformation_matrix': [[0.999398059318965, -0.00020282846028934223, -1243.5138093608236], [0.00024455038200216076, 0.9999230769021725, -32.762125098451975], [0.0, 0.0, 1.0]], 'binary_mask_excluded': False, 'test_run_class': 'cfg_0'}

Last job:
{'pair_id': 0, 'fixed_path': 'data/images/6045.data', 'movi